# VLA: from observations to actions

Action chunks, masks, normalization, and supported policy families.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

A VLA consumes observations, instructions, and state where supported. Outputs may be action tokens, a vector, or a chunk of future actions. Objectives depend on the policy: OpenVLA uses action-token CE, ACT reconstruction plus KL, and SmolVLA/π0 flow matching.

These standalone policies use demonstrations in Studio, not universal GRPO/Unsloth. RL for Studio’s native VLA is a separate route. This exercise is only masked continuous imitation, not ACT or π0. Terminal padding is not real action data. Hardware use additionally requires verified units, calibration, and action limits.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
batch, horizon, action_dim = 24, 4, 3
state = torch.randn(batch, 6)
target = state[:, :3, None].transpose(1, 2).repeat(1, horizon, 1) * 0.2
target += torch.arange(horizon)[None, :, None] * 0.05
valid = torch.ones(batch, horizon, dtype=torch.bool); valid[:8, -2:] = False
mean = target[valid].mean(0); std = target[valid].std(0).clamp_min(1e-6)
normalized = (target - mean) / std
policy = nn.Sequential(nn.Linear(6, 32), nn.ReLU(), nn.Linear(32, horizon * action_dim))
optimizer = torch.optim.AdamW(policy.parameters(), lr=0.02)
for step in range(60):
    prediction = policy(state).reshape(batch, horizon, action_dim)
    loss = ((prediction - normalized).square() * valid[..., None]).sum() / (valid.sum() * action_dim)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
action = prediction.detach() * std + mean
print('valid steps:', valid.sum().item(), 'loss:', loss.item())
print('first chunk in original units:', action[0])

## Try it yourself

Try horizons 1 and 8. Remove the mask: how does the objective change? In real training, compute normalization on training data only and export it with the policy.

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [OpenVLA](https://arxiv.org/abs/2406.09246)
- [ACT](https://arxiv.org/abs/2304.13705)
- [π0](https://arxiv.org/abs/2410.24164)